
# Paper 1 fixed-spec five-seed UQ confirmation

**Purpose:** confirm stochastic stability of the already-frozen Paper 1 uncertainty results.

**Seeds:** 42, 123, 456, 789, 1000.

**Frozen choices:** no architecture selection, no hyperparameter tuning, no split changes, no calibration-rule changes, no DS-A/DS-B threshold changes. The final quantile model uses 25 epochs, batch size 64, Adam 1e-3, the 20→18 linear projection, BiLSTM(64 each direction), temporal attention, and q10/q50/q90 pinball loss.

**Methods reported:** Asymmetric CQR and Balanced Engine CQR.

**Metrics:** PICP, MPIW, Winkler score, point RMSE/MAE, calibration corrections. Outputs are written per seed and as mean ± sample SD.

**Quantile convention preserved from the frozen notebooks:** finite-sample conformal quantiles use `method="higher"`. The first-stage within-engine 0.80 quantile in Balanced Engine CQR intentionally uses NumPy's default interpolation, matching the frozen implementation; the second-stage across-engine conformal quantile uses `method="higher"`.

Seed 42 is treated as a reproducibility check against the frozen master result. A mismatch is a warning to stop and resolve the runtime/data specification before using the multi-seed summary.


## IID / standard-condition confirmation
This notebook is self-contained if `FD004_baseline_tensors.npz` is in the usual Paper 1 artifact location.

In [4]:

import os, random, time, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Layer
from tensorflow.keras.models import Model

SEEDS = [42, 123, 456, 789, 1000]
ALPHA = 0.20
QUANTILES = [0.10, 0.50, 0.90]
EPOCHS = 25
BATCH_SIZE = 64

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def conformal_quantile(scores, alpha):
    scores = np.asarray(scores)
    n = len(scores)
    q_level = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(scores, q_level, method="higher"))

def interval_metrics(y, lo, hi, alpha=ALPHA):
    y = np.asarray(y)
    lo = np.asarray(lo)
    hi = np.asarray(hi)
    covered = (y >= lo) & (y <= hi)
    width = hi - lo
    winkler = (
        width
        + (2.0 / alpha) * (lo - y) * (y < lo)
        + (2.0 / alpha) * (y - hi) * (y > hi)
    )
    return {
        "PICP": 100.0 * covered.mean(),
        "MPIW": width.mean(),
        "Winkler": winkler.mean(),
    }

def summarize_seed_table(df):
    numeric_cols = [
        c for c in df.columns
        if c not in ("Condition", "Method", "Seed")
        and pd.api.types.is_numeric_dtype(df[c])
    ]
    rows = []
    for (condition, method), g in df.groupby(["Condition","Method"], sort=False):
        row = {"Condition": condition, "Method": method, "N_Seeds": len(g)}
        for c in numeric_cols:
            row[c + "_mean"] = g[c].mean()
            row[c + "_sd"] = g[c].std(ddof=1)
        rows.append(row)
    return pd.DataFrame(rows)

def out_dir():
    drive_dir = "/content/drive/MyDrive/FD004_Research/Paper1_MultiSeed_UQ"
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs(drive_dir, exist_ok=True)
        return drive_dir
    local = "/content/Paper1_MultiSeed_UQ"
    os.makedirs(local, exist_ok=True)
    return local

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Output directory:", out_dir())


TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Output directory: /content/drive/MyDrive/FD004_Research/Paper1_MultiSeed_UQ


In [5]:

# Load frozen baseline tensors.
candidate_paths = [
    "/content/FD004_baseline_tensors.npz",
    "/content/drive/MyDrive/FD004_Research/artifacts/FD004_baseline_tensors.npz",
]
DATA_PATH = next((p for p in candidate_paths if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "FD004_baseline_tensors.npz not found. Upload it to /content or place it in "
        "/content/drive/MyDrive/FD004_Research/artifacts/."
    )

with np.load(DATA_PATH) as d:
    X_train_raw = d["X_train"]
    y_train = d["y_train"]
    X_val_raw = d["X_val"]
    y_val = d["y_val"]

# Frozen final representation: 14 sensors + 6 one-hot regimes = 20 channels.
X_tr_sens = X_train_raw[:, :, :14]
X_val_sens = X_val_raw[:, :, :14]
reg_tr = X_train_raw[:, :, 14].astype(int)
reg_val = X_val_raw[:, :, 14].astype(int)

X_tr_oh = np.concatenate(
    [X_tr_sens, tf.keras.utils.to_categorical(reg_tr, num_classes=6)],
    axis=-1
)
X_val_oh = np.concatenate(
    [X_val_sens, tf.keras.utils.to_categorical(reg_val, num_classes=6)],
    axis=-1
)

# Reconstruct the 49 validation-engine labels exactly as in the frozen notebook.
val_engine_ids = np.zeros(len(y_val), dtype=int)
curr_eng = 1
val_engine_ids[0] = curr_eng
for i in range(1, len(y_val)):
    if y_val[i] > y_val[i-1]:
        curr_eng += 1
    val_engine_ids[i] = curr_eng

unique_engines = np.unique(val_engine_ids)

# Frozen 25 calibration / 24 evaluation engine split.
np.random.seed(42)
cal_engines = np.random.choice(unique_engines, size=25, replace=False)
eval_engines = np.setdiff1d(unique_engines, cal_engines)

cal_mask = np.isin(val_engine_ids, cal_engines)
eval_mask = np.isin(val_engine_ids, eval_engines)

X_cal = X_val_oh[cal_mask]
y_cal = y_val[cal_mask]
ids_cal = val_engine_ids[cal_mask]

X_eval = X_val_oh[eval_mask]
y_eval = y_val[eval_mask]
ids_eval = val_engine_ids[eval_mask]

print("Train:", X_tr_oh.shape, "Cal:", X_cal.shape, "Eval:", X_eval.shape)
print("Calibration engines:", len(np.unique(ids_cal)))
print("Evaluation engines:", len(np.unique(ids_eval)))
assert X_tr_oh.shape[-1] == 20
assert len(np.unique(ids_cal)) == 25
assert len(np.unique(ids_eval)) == 24


Train: (43696, 30, 20) Cal: (4909, 30, 20) Eval: (5423, 30, 20)
Calibration engines: 25
Evaluation engines: 24


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:

class FastTemporalAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    def build(self, input_shape):
        dim = input_shape[-1]
        self.W = self.add_weight(name="W", shape=(dim, dim), initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(name="b", shape=(dim,), initializer="zeros", trainable=True)
        self.v = self.add_weight(name="v", shape=(dim, 1), initializer="glorot_uniform", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        e = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        scores = tf.tensordot(e, self.v, axes=1)
        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(inputs * weights, axis=1)

def tilted_loss(q, y_true, y_pred):
    err = y_true - y_pred
    return tf.maximum(q * err, (q - 1.0) * err)

def multi_quantile_pinball_loss(y_true, y_pred):
    y_t = tf.cast(tf.reshape(y_true, (-1, 1)), tf.float32)
    losses = [tilted_loss(q, y_t, y_pred[:, i:i+1]) for i,q in enumerate(QUANTILES)]
    return tf.reduce_mean(tf.add_n(losses))

def build_model():
    inp = Input(shape=(30,20), name="input_window")
    x = Dense(18, use_bias=False, name="projection")(inp)
    x = Bidirectional(LSTM(64, return_sequences=True), name="bilstm")(x)
    x = FastTemporalAttention(name="temporal_attention")(x)
    out = Dense(3, activation="linear", name="quantiles")(x)
    m = Model(inp, out, name="FD004_Paper1_Final_Quantile")
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=multi_quantile_pinball_loss)
    return m


In [7]:

records = []
for run_i, seed in enumerate(SEEDS, 1):
    print(f"\n[{run_i}/{len(SEEDS)}] IID seed {seed}")
    tf.keras.backend.clear_session()
    gc.collect()
    set_seed(seed)
    model = build_model()
    model.fit(X_tr_oh, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2, shuffle=True)

    cal_pred = model.predict(X_cal, batch_size=512, verbose=0)
    test_pred = model.predict(X_eval, batch_size=512, verbose=0)
    p10c,p50c,p90c = cal_pred[:,0],cal_pred[:,1],cal_pred[:,2]
    p10,p50,p90 = test_pred[:,0],test_pred[:,1],test_pred[:,2]

    # Asymmetric CQR: preserve frozen side-wise alpha/2 convention.
    q_lo = conformal_quantile(p10c - y_cal, ALPHA/2)
    q_hi = conformal_quantile(y_cal - p90c, ALPHA/2)
    lo_a = np.maximum(0.0, p10 - q_lo)
    hi_a = np.maximum(0.0, p90 + q_hi)

    # Balanced Engine CQR: within-engine q80 uses NumPy default; across engines uses "higher".
    cal_err = np.maximum(p10c - y_cal, y_cal - p90c)
    eng_q80 = np.asarray([
        np.quantile(cal_err[ids_cal == eng], 0.80)
        for eng in np.unique(ids_cal)
    ])
    n_eng = len(eng_q80)
    q_level_eng = min(1.0, np.ceil((n_eng + 1) * (1 - ALPHA)) / n_eng)
    q_bal = float(np.quantile(eng_q80, q_level_eng, method="higher"))
    lo_b = np.maximum(0.0, p10 - q_bal)
    hi_b = np.maximum(0.0, p90 + q_bal)

    rmse = float(np.sqrt(mean_squared_error(y_eval, p50)))
    mae = float(mean_absolute_error(y_eval, p50))

    for method, lo, hi, qa, qb in [
        ("Asymmetric CQR", lo_a, hi_a, q_lo, q_hi),
        ("Balanced Engine CQR", lo_b, hi_b, q_bal, q_bal),
    ]:
        met = interval_metrics(y_eval, lo, hi)
        records.append({
            "Condition":"IID","Seed":seed,"Method":method,
            "PICP":met["PICP"],"MPIW":met["MPIW"],"Winkler":met["Winkler"],
            "RMSE":rmse,"MAE":mae,
            "Q_lower":qa,"Q_upper":qb
        })
    print(pd.DataFrame(records[-2:])[["Seed","Method","PICP","MPIW","Winkler","RMSE"]].round(3).to_string(index=False))

iid_seed = pd.DataFrame(records)
iid_summary = summarize_seed_table(iid_seed)

od = out_dir()
iid_seed.to_csv(os.path.join(od, "Paper1_FiveSeed_IID_per_seed.csv"), index=False)
iid_summary.to_csv(os.path.join(od, "Paper1_FiveSeed_IID_summary.csv"), index=False)

display(iid_seed.round(3))
display(iid_summary.round(3))
print("Saved IID outputs to", od)



[1/5] IID seed 42
Epoch 1/25
683/683 - 14s - 20ms/step - loss: 74.2914
Epoch 2/25
683/683 - 7s - 10ms/step - loss: 32.0112
Epoch 3/25
683/683 - 10s - 14ms/step - loss: 28.8946
Epoch 4/25
683/683 - 7s - 11ms/step - loss: 28.8822
Epoch 5/25
683/683 - 9s - 14ms/step - loss: 25.6487
Epoch 6/25
683/683 - 7s - 10ms/step - loss: 15.5320
Epoch 7/25
683/683 - 6s - 9ms/step - loss: 13.3338
Epoch 8/25
683/683 - 7s - 10ms/step - loss: 12.4569
Epoch 9/25
683/683 - 6s - 9ms/step - loss: 12.0021
Epoch 10/25
683/683 - 10s - 15ms/step - loss: 11.6469
Epoch 11/25
683/683 - 7s - 10ms/step - loss: 11.3079
Epoch 12/25
683/683 - 9s - 13ms/step - loss: 11.1065
Epoch 13/25
683/683 - 7s - 10ms/step - loss: 10.8615
Epoch 14/25
683/683 - 6s - 8ms/step - loss: 10.6572
Epoch 15/25
683/683 - 7s - 10ms/step - loss: 10.4178
Epoch 16/25
683/683 - 6s - 9ms/step - loss: 10.1311
Epoch 17/25
683/683 - 8s - 11ms/step - loss: 9.9276
Epoch 18/25
683/683 - 6s - 9ms/step - loss: 9.7582
Epoch 19/25
683/683 - 7s - 11ms/step - l

,Condition,Seed,Method,PICP,MPIW,Winkler,RMSE,MAE,Q_lower,Q_upper
0,IID,42,Asymmetric CQR,79.919,41.569000,70.401001,21.458,13.314,9.650,4.659
1,IID,42,Balanced Engine CQR,91.573,67.498001,77.068001,21.458,13.314,20.770,20.770
2,IID,123,Asymmetric CQR,84.603,57.810001,83.013000,24.342,15.956,4.798,15.327
3,IID,123,Balanced Engine CQR,91.093,79.157997,92.026001,24.342,15.956,21.899,21.899
4,IID,456,Asymmetric CQR,82.095,41.513000,68.375999,20.985,12.856,13.156,3.040
5,IID,456,Balanced Engine CQR,93.933,73.559998,81.300003,20.985,12.856,24.811,24.811
6,IID,789,Asymmetric CQR,83.367,47.662998,76.794998,22.696,14.257,9.924,9.290
7,IID,789,Balanced Engine CQR,93.601,81.885002,91.181000,22.696,14.257,27.842,27.842
8,IID,1000,Asymmetric CQR,79.937,41.514999,79.355003,23.805,14.452,10.564,3.602
9,IID,1000,Balanced Engine CQR,91.352,75.125999,88.301003,23.805,14.452,24.821,24.821


,Condition,Method,N_Seeds,PICP_mean,PICP_sd,MPIW_mean,MPIW_sd,Winkler_mean,Winkler_sd,RMSE_mean,RMSE_sd,MAE_mean,MAE_sd,Q_lower_mean,Q_lower_sd,Q_upper_mean,Q_upper_sd
0,IID,Asymmetric CQR,5,81.984,2.076,46.014,7.108,75.587997,6.118,22.657,1.448,14.167,1.197,9.618,3.031,7.183,5.175
1,IID,Balanced Engine CQR,5,92.311,1.346,75.445,5.524,85.974998,6.525,22.657,1.448,14.167,1.197,24.029,2.781,24.029,2.781


Saved IID outputs to /content/drive/MyDrive/FD004_Research/Paper1_MultiSeed_UQ
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [8]:

# Seed-42 frozen-master gate.
EXPECTED = {
    "Asymmetric CQR": {"PICP":83.24, "MPIW":51.50, "Winkler":76.03},
    "Balanced Engine CQR": {"PICP":91.67, "MPIW":72.30, "Winkler":84.13},
}
TOL = {"PICP":0.75, "MPIW":1.50, "Winkler":2.00}

s42 = iid_seed[iid_seed["Seed"] == 42].set_index("Method")
gate_rows = []
for method, vals in EXPECTED.items():
    row = {"Method":method}
    ok = True
    for metric, exp in vals.items():
        obs = float(s42.loc[method, metric])
        diff = obs-exp
        within = abs(diff) <= TOL[metric]
        row.update({metric+"_observed":obs, metric+"_expected":exp, metric+"_diff":diff, metric+"_PASS":within})
        ok &= within
    row["Overall_PASS"] = ok
    gate_rows.append(row)
gate = pd.DataFrame(gate_rows)
display(gate)
gate.to_csv(os.path.join(out_dir(),"Paper1_FiveSeed_IID_seed42_gate.csv"), index=False)
if not bool(gate["Overall_PASS"].all()):
    print("STOP: seed-42 IID does not reproduce the frozen master closely enough. Do not use the summary yet.")
else:
    print("PASS: seed-42 IID reproduction gate passed.")


,Method,PICP_observed,PICP_expected,PICP_diff,PICP_PASS,MPIW_observed,MPIW_expected,MPIW_diff,MPIW_PASS,Winkler_observed,Winkler_expected,Winkler_diff,Winkler_PASS,Overall_PASS
0,Asymmetric CQR,79.918864,83.24,-3.321136,False,41.569454,51.5,-9.930546,False,70.400726,76.03,-5.629274,False,False
1,Balanced Engine CQR,91.572930,91.67,-0.097070,True,67.497513,72.3,-4.802487,False,77.067696,84.13,-7.062304,False,False


STOP: seed-42 IID does not reproduce the frozen master closely enough. Do not use the summary yet.
